Data komentar (query + text)

↓
Preprocessing teks (cleaning, stopword, dll)

↓
Pelabelan sentimen (Positif / Negatif / Netral)  ← perlu dilakukan manual/semi-otomatis

↓
Training model deep learning (LSTM / IndoBERT)

↓
Evaluasi akurasi training & testing set

## Preprocessing Data
Memulai pemrosesan data untuk youtube_dataset.json


In [28]:
import pandas as pd
import json
import re
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from nltk.tokenize import word_tokenize
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

import emoji


In [2]:
# Load data
with open("youtube_dataset.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Jadikan DataFrame dan ambil hanya kolom query + text
df = pd.DataFrame(data)[["query", "text"]]

In [4]:
# cek apakah ada baris kosong
print(df[df["text"].str.strip() == ""])

df.isnull().sum()

Empty DataFrame
Columns: [query, text]
Index: []


query    0
text     0
dtype: int64

In [ ]:
# 
df = df.reset_index(drop=True)

print(f"Total data: {len(df)}")
print(df.head())

Total data: 82509
                      query                                               text
0  korban menjadi tersangka  Sudah jelas ya. Penegak hukum,polisi,Jaksa,hak...
1  korban menjadi tersangka                     parah nya hukum Indonesia😂😂😂😂😂
2  korban menjadi tersangka  Hukum dan penegak hukum memang bajingan di neg...
3  korban menjadi tersangka          Parah sih emang penegak hukum negeri kita
4  korban menjadi tersangka      Rakyat indonesia sudah amat geram sama polisi


Teks Mentah → Cleaning → Case Folding → Normalisasi → Tokenisasi → Stopword Removal → Stemming → Teks Bersih

In [35]:
# Cleaning data
# def clean_emoji(text):
#     return emoji.replace_emoji(text, replace="")

def cleaning(text):
    text = emoji.replace_emoji(text, replace="")
    text = re.sub(r'http\S+|www\S+', '', text)      # hapus URL
    text = re.sub(r'@\w+', '', text)                # hapus mention
    text = re.sub(r'#\w+', '', text)                # hapus hashtag
    text = re.sub(r'\d+', '', text)                 # hapus angka
    text = re.sub(r'[^\w\s]', '', text)             # hapus tanda baca
    text = re.sub(r'\s+', ' ', text).strip()        # normalisasi spasi
    # menghapus angka
    text = re.sub(r'\d+', '', text)
    return text

In [36]:
df["text_clean"] = df["text"].apply(cleaning)
# hapus kolom cleaned_text jika tidak diperlukan
# df = df.drop(columns=["cleaned_text"])
df.head()

,query,text,text_clean
0,korban menjadi tersangka,"Sudah jelas ya. Penegak hukum,polisi,Jaksa,hak...",Sudah jelas ya Penegak hukumpolisiJaksahakimda...
1,korban menjadi tersangka,parah nya hukum Indonesia😂😂😂😂😂,parah nya hukum Indonesia
2,korban menjadi tersangka,Hukum dan penegak hukum memang bajingan di neg...,Hukum dan penegak hukum memang bajingan di neg...
3,korban menjadi tersangka,Parah sih emang penegak hukum negeri kita,Parah sih emang penegak hukum negeri kita
4,korban menjadi tersangka,Rakyat indonesia sudah amat geram sama polisi,Rakyat indonesia sudah amat geram sama polisi


In [34]:
# test emoji
print(emoji.replace_emoji("parah nya hukum Indonesia😂😂😂😂😂", replace=''))
print("parah nya hukum Indonesia😂😂😂😂😂")

parah nya hukum Indonesia
parah nya hukum Indonesia😂😂😂😂😂


In [37]:
# hapus kolom text jika tidak diperlukan
df = df.drop(columns=["text"])
df.head()

,query,text_clean
0,korban menjadi tersangka,Sudah jelas ya Penegak hukumpolisiJaksahakimda...
1,korban menjadi tersangka,parah nya hukum Indonesia
2,korban menjadi tersangka,Hukum dan penegak hukum memang bajingan di neg...
3,korban menjadi tersangka,Parah sih emang penegak hukum negeri kita
4,korban menjadi tersangka,Rakyat indonesia sudah amat geram sama polisi


In [38]:
# case folding menjadi huruf kecil semua
def case_folding(text):
    return text.lower()

df["text_clean"] = df["text_clean"].apply(case_folding)
df.head()

,query,text_clean
0,korban menjadi tersangka,sudah jelas ya penegak hukumpolisijaksahakimda...
1,korban menjadi tersangka,parah nya hukum indonesia
2,korban menjadi tersangka,hukum dan penegak hukum memang bajingan di neg...
3,korban menjadi tersangka,parah sih emang penegak hukum negeri kita
4,korban menjadi tersangka,rakyat indonesia sudah amat geram sama polisi


In [ ]:
# normalisasi kata tidak baku / slang

# mendownload normalisasi
url = "https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv"
kamus_df = pd.read_csv(url)
kamus_df.to_csv
normalization_dict = {
    "gak": "tidak", "ga": "tidak", "nggak": "tidak", "ngga": "tidak",
    "yg": "yang", "dgn": "dengan", "utk": "untuk", "krn": "karena",
    "udah": "sudah", "udh": "sudah", "sdh": "sudah",
    "aja": "saja", "aj": "saja",
    "kalo": "kalau", "klo": "kalau",
    "gue": "saya", "gw": "saya", "aku": "saya",
    "lu": "kamu", "lo": "kamu",
    "bgt": "banget", "bngt": "banget",
    "tdk": "tidak", "tdk": "tidak", "tdk": "tidak",
    "jg": "juga", "juga": "juga",
    "tp": "tapi", "tpi": "tapi",
    "sm": "sama", "sama": "sama",
    "ny": "nya", "emg": "memang",
    "hrs": "harus", "hrs": "harus",
    "org": "orang", "orng": "orang",
    "pd": "pada", "dr": "dari", "ke": "ke",
    "sy": "saya", "mrk": "mereka",
    "bs": "bisa", "bsa": "bisa",
    "sdg": "sedang", "lg": "lagi",
    "kpd": "kepada", "spy": "supaya",
}

def normalize(text):
    words = text.split()
    words = [normalization_dict.get(word, word) for word in words]
    return ' '.join(words)

In [ ]:
# Stopword removal - Menghapus kata-kata umum yang tidak memiliki makna penting
factory = StopWordRemoverFactory()
stopword_remover = factory.create_stop_word_remover()

def remove_stopwords(text):
    return stopword_remover.remove(text)

⚠️ Catatan: Untuk deep learning (LSTM/BERT), stemming kadang tidak wajib bahkan bisa mengurangi akurasi. IndoBERT bekerja lebih baik tanpa stemming karena sudah memahami imbuhan. Stemming lebih berguna untuk TF-IDF + SVM/RF.

In [15]:
# Stemming - Mengubah kata berimbuhan ke kata dasar (stemming)

factory = StemmerFactory()
stemmer = factory.create_stemmer()

def stemming(text):
    return stemmer.stem(text)

In [ ]:
# Tokenisasi - Memecah teks menjadi kata-kata atau token
def tokenization(text):
    # atau pakai NLTK:
    # from nltk.tokenize import word_tokenize
    return word_tokenize(text)

,query,text,text_clean
0,korban menjadi tersangka,"Sudah jelas ya. Penegak hukum,polisi,Jaksa,hak...",Sudah jelas ya Penegak hukumpolisiJaksahakimda...
1,korban menjadi tersangka,parah nya hukum Indonesia😂😂😂😂😂,parah nya hukum Indonesia
2,korban menjadi tersangka,Hukum dan penegak hukum memang bajingan di neg...,Hukum dan penegak hukum memang bajingan di neg...
3,korban menjadi tersangka,Parah sih emang penegak hukum negeri kita,Parah sih emang penegak hukum negeri kita
4,korban menjadi tersangka,Rakyat indonesia sudah amat geram sama polisi,Rakyat indonesia sudah amat geram sama polisi


In [23]:
# diproses semua
def full_preprocess(text):
    text = str(text)
    text = cleaning(text)
    text = case_folding(text)
    text = normalize(text)
    text = remove_stopwords(text)
    text = stemming(text)   # pindahkan ke sini
    return text

df['text_clean'] = df['text'].apply(full_preprocess)

print(df[['text', 'text_clean']].head(10))

KeyboardInterrupt: 